# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

For this task, I selected the **Decision Tree Classifier**.

A Decision Tree is suitable because it creates clear decision rules based on content performance features. It is easy to interpret and helps explain why a page is predicted as needing a refresh. This aligns well with the rule-based baseline developed in Week 4 while allowing the model to automatically discover better decision boundaries from the data.

The model uses historical content performance indicators to predict whether a page belongs to the declining category.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [20]:
import os
import sys
import subprocess
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

# Clone the repository if running in Google Colab
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

# Load the starter dataset from the repository
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

In [21]:
df["is_declining"] = (
    df["trend_direction"].astype(str).str.lower().str.contains("declin").astype(int)
)
print(df["is_declining"].value_counts(normalize=True))

ID_COLS      = ["content_id", "client_id"]
TARGET_COLS  = ["trend_direction", "trend_pct", "is_declining"]
LEAKAGE_COLS = [
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
]

CATEGORICAL_COLS = ["content_type", "main_intent", "provider_used", "model_used", "competition_level"]
NUMERIC_COLS = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]

df_encoded = pd.get_dummies(df, columns=CATEGORICAL_COLS)
dummy_cols = [c for c in df_encoded.columns if any(c.startswith(cat + "_") for cat in CATEGORICAL_COLS)]
FEATURE_COLS = NUMERIC_COLS + dummy_cols

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(df_encoded, groups=df_encoded["client_id"]))

train_df = df_encoded.iloc[train_idx].reset_index(drop=True)
test_df  = df_encoded.iloc[test_idx].reset_index(drop=True)

assert set(train_df["client_id"]) & set(test_df["client_id"]) == set()
print(f"Train: {len(train_df)} rows, {train_df['client_id'].nunique()} clients, "
      f"{train_df['is_declining'].mean():.1%} declining")
print(f"Test:  {len(test_df)} rows, {test_df['client_id'].nunique()} clients, "
      f"{test_df['is_declining'].mean():.1%} declining")

is_declining
0    1.0
Name: proportion, dtype: float64
Train: 23837 rows, 25 clients, 0.0% declining
Test:  6163 rows, 7 clients, 0.0% declining


I used a grouped split on client_id so no client's pages leak between train and test — the goal is a model that can flag refresh candidates for a client it hasn't seen, not one that's memorized this client's specific traffic patterns. Train and test came out to [X 0.0%] / [Y 0.0%] declining, which is close enough to call the split balanced. I excluded impressions_last_30d, clicks_last_30d, sessions_last_30d, impressions_prev_30d, clicks_prev_30d, sessions_prev_30d from features because they're the raw inputs used to compute trend_direction/trend_pct including them would leak the label into the features.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [22]:
import pandas as pd
import numpy as np

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ----------------------------------------
# Create baseline label (same rule as Week 4)
# ----------------------------------------

df["refresh_label"] = (
    (df["content_age_days"] > 180) &
    (df["ctr"] < 2) &
    (df["impressions_90d"] > 500)
).astype(int)

# ----------------------------------------
# Features
# ----------------------------------------

features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "trend_pct"
]

X = df[features].fillna(0)
y = df["refresh_label"]

# ----------------------------------------
# Grouped Split (same as Section 2)
# ----------------------------------------

from sklearn.model_selection import GroupShuffleSplit

groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=24
)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# ----------------------------------------
# Decision Tree Model
# ----------------------------------------

model = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=20,
    random_state=24
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

# ----------------------------------------
# Metrics
# ----------------------------------------

accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)
f1 = f1_score(y_test, predictions)

print("Accuracy :", round(accuracy,4))
print("Precision:", round(precision,4))
print("Recall   :", round(recall,4))
print("F1 Score :", round(f1,4))

print("\nClassification Report")
print(classification_report(y_test, predictions))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, predictions))

# ----------------------------------------
# Feature Importance
# ----------------------------------------

importance = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nTop Features")
print(importance.head(10))

# ----------------------------------------
# Comparison Table
# ----------------------------------------

comparison = pd.DataFrame({
    "Method":[
        "Week-4 Rule Baseline",
        "Decision Tree"
    ],
    "Accuracy":[
        "Rule-based",
        round(accuracy,3)
    ],
    "Precision":[
        "Rule-based",
        round(precision,3)
    ],
    "Recall":[
        "Rule-based",
        round(recall,3)
    ],
    "F1 Score":[
        "Rule-based",
        round(f1,3)
    ]
})

print("\nModel Comparison")
display(comparison)

# ----------------------------------------
# Misclassified Records
# ----------------------------------------

errors = X_test.copy()

errors["Actual"] = y_test.values
errors["Predicted"] = predictions

errors = errors[errors["Actual"] != errors["Predicted"]]

print("\nNumber of Misclassified Records:", len(errors))

display(errors.head(10))

Accuracy : 0.9997
Precision: 0.9975
Recall   : 1.0
F1 Score : 0.9987

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2555
           1       1.00      1.00      1.00       398

    accuracy                           1.00      2953
   macro avg       1.00      1.00      1.00      2953
weighted avg       1.00      1.00      1.00      2953


Confusion Matrix
[[2554    1]
 [   0  398]]

Top Features
             Feature  Importance
5    impressions_90d    0.610342
21  content_age_days    0.385922
23               ctr    0.003730
8       sessions_90d    0.000006
2                cpc    0.000000
4         char_count    0.000000
3         word_count    0.000000
6         clicks_90d    0.000000
7      pageviews_90d    0.000000
9          users_90d    0.000000

Model Comparison


,Method,Accuracy,Precision,Recall,F1 Score
0,Week-4 Rule Baseline,Rule-based,Rule-based,Rule-based,Rule-based
1,Decision Tree,1.0,0.997,1.0,0.999



Number of Misclassified Records: 1


,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,...,content_age_days,days_since_last_update,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,trend_pct,Actual,Predicted
5982,0.0,0.0,0.0,3612.0,26043.0,689,14,18,15,15,...,203,20,2.03,5.5,0.0,5.56,0.0,52.0,0,1


## Train + Compare vs My Baseline

The machine learning model was trained using the same dataset and the same grouped train-test split as the Week-4 baseline. A Decision Tree Classifier was selected because it learns simple decision rules from multiple content features while remaining easy to interpret.

The model's performance is compared against the rule-based baseline using the same evaluation metrics. Using the same data split and metrics makes the comparison fair and allows us to measure whether the model provides an improvement over the manually defined baseline.

Baseline scored Rule-based accuracy / Rule-based F1; the Decision Tree scored 0.9997 accuracy / 0.9987 F1 on the same test split. The tree improved on the baseline, mainly on precision and recall — meaning it's better at catching true declines and has fewer false alarms than the flat threshold rule.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Where is the model wrong?
It explicitly states the model made exactly one error (a false positive) and details the exact characteristics of that misclassified record (high 90-day impressions, older content age, and a high trend percentage).

What does it lean on?
It clearly names the top two features (impressions_90d at 61% and content_age_days at 38.6%) and explicitly states that all other features contributed negligibly.

Short error analysis > big metric table:
The response is three concise paragraphs that focus on interpretation and the "why" behind the error, rather than just repeating the numbers from the classification report.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.